# GISR-Net — revision experiments on Google Colab

Runs the complete study behind the revised manuscript:

* six **source-image-disjoint** folds, exact 80/10/10 split (Reviewer 1.2 / 1.4, Reviewer 2.1 / 2.2)
* external validation on **MED-NODE**, an independent database and camera source (Reviewer 1.5)
* an expanded ablation study (Reviewer 1.7)
* every figure, table and both Word documents, regenerated from the real numbers

**Before you start:** *Runtime → Change runtime type → Hardware accelerator → **T4 GPU***.

Expected runtime on a T4: roughly **35–60 minutes** for all five configurations.

## 1. Check the runtime

In [11]:
import torch, platform
print("Python  ", platform.python_version())
print("PyTorch ", torch.__version__)
if torch.cuda.is_available():
    print("GPU     ", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
else:
    print("GPU      NONE  ->  Runtime > Change runtime type > T4 GPU")
    print("         (it still runs on CPU, but expect several hours)")

Python   3.12.13
PyTorch  2.11.0+cu128
GPU      Tesla T4 (15.6 GB)


## 2. Install the one missing dependency

Colab already ships PyTorch, NumPy, SciPy, Matplotlib, pandas and Pillow. Only `python-docx` is needed, for the Word output.

In [12]:
!pip install -q python-docx
print("done")

done


## 3. Get the data into Colab

> ### ⚠️ If your folder lives under **Computers → My Mac** in Google Drive
>
> Colab's `drive.mount()` exposes **only "My Drive"**. Folders under the
> **Computers** section (Google Drive for Desktop backup) are *not* visible to Colab.
>
> **Fix — takes about 10 seconds:** in Google Drive on the web, right-click
> **GISRNET_Work** under *Computers → My Mac → Desktop* → **Organise** →
> **Add shortcut to Drive** → choose **My Drive** → *Add*.
>
> The shortcut costs no extra storage and Colab resolves it normally. Cell 3b below
> finds it automatically.
>
> If shortcuts give you trouble, use **Option B** further down instead.

### Option A — mount Google Drive (recommended)

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 3b. Locate `GISRNET_Work` automatically

Searches My Drive (and any shortcuts) for a folder containing `Data.csv`.
Set `DRIVE_PATH` manually below only if the search comes up empty.

In [14]:
# Self-contained: finds GISRNET_Work anywhere in the mounted Drive.
# Set DRIVE_PATH only if you want to force a specific location.
import os, pathlib

DRIVE_PATH = None          # e.g. '/content/drive/MyDrive/GISRNET_Work'

MARKERS = ("Data.csv", "HF_Test.csv")
BASES = ("/content/drive/MyDrive", "/content/drive/My Drive",
         "/content/drive/Shareddrives", "/content/drive/Shared drives",
         "/content/drive/.shortcut-targets-by-id", "/content/data", "/content")
SKIP = {".git", "__pycache__", ".ipynb_checkpoints", "sample_data",
        "GISRNet_Results", "node_modules", ".cache", ".config", "drive"}

def is_root(p):
    try:
        return all((p / m).exists() for m in MARKERS)
    except OSError:
        return False

def search(max_depth=4):
    hits = []
    for base in BASES:
        b = pathlib.Path(base)
        if not b.exists():
            continue
        print(f"  searching {base} ...")
        nparts = len(b.parts)
        for dirpath, dirnames, _ in os.walk(b, onerror=lambda e: None, followlinks=True):
            p = pathlib.Path(dirpath)
            if len(p.parts) - nparts >= max_depth:
                dirnames[:] = []
            else:
                dirnames[:] = [d for d in dirnames if d not in SKIP and not d.startswith(".")]
            if is_root(p) and p not in hits:
                hits.append(p)
    return sorted(hits, key=lambda p: len(str(p)))

def diagnose():
    print("\n" + "=" * 74)
    print("Could not find a folder containing both Data.csv and HF_Test.csv.")
    print("=" * 74)
    drive = pathlib.Path("/content/drive")
    if not drive.exists():
        print("\nGoogle Drive is NOT mounted -- run the drive.mount() cell above first.")
        return
    print("\nTop level of /content/drive:")
    for p in sorted(drive.iterdir()):
        print("   ", p.name + ("/" if p.is_dir() else ""))
    for cand in ("/content/drive/MyDrive", "/content/drive/My Drive"):
        d = pathlib.Path(cand)
        if not d.is_dir():
            continue
        print(f"\nFolders in {cand}:")
        try:
            entries = sorted(x for x in d.iterdir() if x.is_dir())
        except OSError as e:
            print("   (could not list:", e, ")"); continue
        for p in entries[:60]:
            print("   ", p.name + "/")
        if len(entries) > 60:
            print(f"    ... and {len(entries) - 60} more")
        break
    print("""
If GISRNET_Work is not listed above, it is stored under "Computers -> My Mac"
(Google Drive for Desktop backup). Colab's drive.mount() exposes ONLY "My Drive";
the Computers section is invisible to it.

Fix -- about ten seconds in the Drive web interface:
    right-click GISRNET_Work  ->  Organise  ->  Add shortcut to Drive
    ->  choose My Drive  ->  Add
then re-run the mount cell and this one.

Alternatives:
  * set DRIVE_PATH above to the exact path, or
  * use Option B (zip upload) below.""")

if DRIVE_PATH:
    root = pathlib.Path(DRIVE_PATH)
    if not is_root(root):
        raise FileNotFoundError(
            f"{root} is missing: "
            + ", ".join(m for m in MARKERS if not (root / m).exists()))
else:
    hits = search()
    if not hits:
        diagnose()
        raise FileNotFoundError("GISRNET_Work not found -- see the guidance above")
    print("\ncandidates found:")
    for h in hits:
        print("   ", h)
    root = hits[0]

print("\nUsing data at:", root)
for d in ("GISRNet_PyTorch", "Enhanced_ECCA", "GT", "HF_Test",
          "Enhanced_MedNodeTest", "MED_NODE_Test_GT", "Simulated_MED_NODE_Test"):
    print(f"  {'OK     ' if (root / d).exists() else 'MISSING'}  {d}")


  searching /content/drive/MyDrive ...
  searching /content/drive/My Drive ...
  searching /content/drive/.shortcut-targets-by-id ...
  searching /content ...

candidates found:
    /content/drive/MyDrive/GISRNET_Work
    /content/drive/My Drive/GISRNET_Work

Using data at: /content/drive/MyDrive/GISRNET_Work
  OK       GISRNet_PyTorch
  OK       Enhanced_ECCA
  OK       GT
  OK       HF_Test
  OK       Enhanced_MedNodeTest
  OK       MED_NODE_Test_GT
  OK       Simulated_MED_NODE_Test


### Option B — upload a zip instead (skip if Option A worked)

Use this if the Drive shortcut is awkward. On your Mac, build a zip of just what the
experiments need — about **224 MB**, versus 1.2 GB for the whole folder:

```bash
cd ~/Desktop
zip -r GISRNET_data.zip GISRNET_Work \
    -x "GISRNET_Work/GISRNet_Results/*" \
    -x "GISRNET_Work/GISRNet_Windows/*" \
    -x "*.docx" -x "*/.DS_Store"
```

Then run the cell below and pick that zip. Note that results will live on Colab's
temporary disk, so **download them before the session ends** (cell 11).

In [ ]:
# Option B only -- leave unrun if Option A already found your data
from google.colab import files
import zipfile, pathlib, os

up = files.upload()                      # choose GISRNET_data.zip
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    z.extractall('/content/data')
cands = [p.parent for p in pathlib.Path('/content/data').rglob('Data.csv')]
assert cands, "no Data.csv inside the zip"
root = cands[0]
print("extracted to", root)

### 3c. Storage routing

Reading 880 JPEGs off Drive every epoch would dominate the runtime, so:

* **code and images** are read from wherever `root` points,
* the **decoded-image cache** goes to local disk (`/content`) — fast, built once in ~30 s,
* **results, checkpoints, figures and documents** are written next to the data, so on Drive
  they survive a disconnect and the run resumes where it stopped.

In [ ]:
import os
os.environ['GISRNET_ROOT']  = str(root)
os.environ['GISRNET_OUT']   = str(root / 'GISRNet_Results')
os.environ['GISRNET_CACHE'] = '/content/gisrnet_cache'      # fast local scratch

CODE = root / 'GISRNet_PyTorch'
assert CODE.exists(), f"code folder not found at {CODE}"
os.chdir(CODE)

import config
print("working dir:", os.getcwd())
print("root       :", config.ROOT)
print("results    :", config.OUT_DIR)
print("cache      :", config.CACHE_DIR)

## 4. Build the folds and prove there is no leakage

This is the answer to the reviewers' central objection. The 50 original photographs are split
into ten blocks of five; each fold takes 40 sources for training, 5 for validation and 5 for
testing — 640/80/80 images. The assertions fail loudly if any photograph appears in two
partitions of the same fold.

In [ ]:
!python folds.py

## 5. Build the external test sets

`HF_Test` (100 images, simulated degradation) and **MED-NODE** (80 images from 80 independent
source photographs). MED-NODE targets are computed here from scratch by running Otsu,
minimum-error and Kapur segmentation against the MED-NODE expert masks — 2–3 minutes.

In [ ]:
!python build_external_sets.py

## 6. Train everything

Five configurations × six folds × up to 30 epochs, with early stopping.

`--auto-lr` selects the initial learning rate on fold 1's validation partition only; the test partitions play no part in the choice.

**Resumable** — if Colab disconnects, re-run cells 3, 3b, 3c and this one. Completed folds are
skipped (as long as results are on Drive).

In [ ]:
import subprocess, time, sys

CONFIGS = [
    ("gisrnet_tl",      ["--arch", "gisrnet"]),
    ("gisrnet_ri",      ["--arch", "gisrnet", "--no-pretrained"]),
    ("gisrnet_noaug",   ["--arch", "gisrnet", "--no-augment"]),
    ("gisrnet_mlp",     ["--arch", "gisrnet", "--head", "mlp"]),
    ("gisrnet_sigmoid", ["--arch", "gisrnet", "--head", "sigmoid"]),
]

EPOCHS  = 30     # set to 10 for a quick end-to-end trial
WORKERS = 2
ACCUM   = 1      # raise to 2 or 4 if you hit out-of-memory

for tag, extra in CONFIGS:
    print(f"\n{'=' * 78}\n>>> {tag}\n{'=' * 78}", flush=True)
    t0 = time.time()
    proc = subprocess.Popen(
        [sys.executable, "-W", "ignore", "train.py", "--tag", tag, "--auto-lr",
         "--workers", str(WORKERS), "--accum", str(ACCUM),
         "--epochs", str(EPOCHS)] + extra,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    print(f"<<< {tag} finished in {(time.time() - t0) / 60:.1f} min "
          f"(exit {proc.returncode})", flush=True)

## 7. Generalisability on the external sets

In [ ]:
!python evaluate_external.py --tags gisrnet_tl

## 8. Figures, tables and the revised Word documents

In [ ]:
!python make_figures.py
!python make_tables.py
!python make_manuscript.py

## 9. Results summary

In [ ]:
import json, pathlib, config

print(f"{'configuration':18s} {'folds':>5s} {'params':>8s} {'lr':>7s} "
      f"{'MAE':>16s} {'RMSE':>8s} {'PLCC':>6s} {'SROCC':>6s} {'KROCC':>6s}")
print("-" * 92)
for p in sorted(pathlib.Path(config.OUT_DIR).glob("results_*.json")):
    if p.name == "results_external.json":
        continue
    r = json.loads(p.read_text())
    m, mo = r.get("mean_test"), r.get("model", {})
    if not m:
        continue
    lr = mo.get("lr_probe", {}).get("selected", mo.get("lr", ""))
    print(f"{p.stem[8:]:18s} {len(r['folds']):5d} {mo.get('params_millions',0):7.2f}M "
          f"{lr:>7} {m['MAE']:8.4f}+-{m['MAE_std']:.4f} {m['RMSE']:8.4f} "
          f"{m['PLCC']:6.3f} {m['SROCC']:6.3f} {m['KROCC']:6.3f}")

## 10. View the figures inline

In [ ]:
from IPython.display import Image, display, Markdown
import pathlib, config

for f in sorted(pathlib.Path(config.FIG_DIR).glob("*.png")):
    display(Markdown(f"### {f.stem}"))
    display(Image(filename=str(f), width=980))

## 11. Download the manuscript and the response letter

If you used Option A they are already saved to Drive. If you used Option B, download them
**before the session ends** — Colab's local disk is wiped on disconnect.

In [ ]:
from google.colab import files
import pathlib, config

for name in ("GISRNet_Revised_Manuscript.docx", "GISRNet_Response_to_Reviewers.docx"):
    p = pathlib.Path(config.ROOT) / name
    if p.exists():
        print("downloading", name, f"({p.stat().st_size / 1024:.0f} KB)")
        files.download(str(p))
    else:
        print("missing:", name)

### Optional — zip the whole results folder for download

In [ ]:
import shutil, config
from google.colab import files
out = shutil.make_archive('/content/GISRNet_Results', 'zip', str(config.OUT_DIR))
print(out, f"{pathlib.Path(out).stat().st_size / 1e6:.0f} MB")
files.download(out)

---

### Troubleshooting

**No candidates found in cell 3b** — your folder is almost certainly under *Computers → My Mac*.
Add a shortcut to My Drive as described at the top of section 3, then re-run cells 3 and 3b.
Failing that, use Option B.

**Out of GPU memory** — set `ACCUM = 2` (or 4) in cell 6. The effective batch size stays 32.

**Colab disconnected mid-run** — re-run cells 3, 3b, 3c, then 6. Folds already recorded in
`results_<tag>.json` are skipped, so nothing is repeated.

**Training is slow** — check that cell 1 reports a GPU. On CPU the full study takes several
hours; set `EPOCHS = 10` in cell 6 for a quicker pass.

**Start over from scratch** — delete `GISRNet_Results/results_*.json`.

**Drive is filling up** — checkpoints are the bulk of it (about 40 MB per fold per
configuration). Delete `GISRNet_Results/models/` once the figures and documents are built;
nothing downstream needs it except `evaluate_external.py`.